# EDA: Raw RBI Regulatory Data

This notebook analyzes the RBI notifications/press releases fetched by
`src/ingestion/rbi_fetcher.py` into `data/raw/*.jsonl`.

**Questions this notebook answers:**
1. How many documents do we have, and from which feeds?
2. How long are documents (word count)? — informs chunking strategy (M3/M4)
3. How reliably does the reference-number pattern (`RBI/20XX-XX/NNN`) appear?
4. How often do documents mention *other* RBI reference numbers (self-referencing
   circulars)? — is multi-hop retrieval actually justified on this corpus?
5. Which regulated-entity categories (Commercial Banks, NBFC, Co-operative Banks,
   etc.) show up, and how are they distributed?
6. What do individual documents actually look like end-to-end?

Run `python -m src.ingestion.rbi_fetcher --feed all` first if `data/raw/` is empty.


In [ ]:
import glob
import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 80)
plt.rcParams["figure.figsize"] = (8, 4)


## 1. Load fetched documents

In [ ]:
files = sorted(glob.glob("../data/raw/*.jsonl"))
print(f"Found {len(files)} fetch file(s):")
for f in files:
    print(" ", f)

records = []
for f in files:
    with open(f, encoding="utf-8") as fh:
        for line in fh:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"\nLoaded {len(df)} total documents.")
df.head()


In [ ]:
# Basic shape and schema check
print(df.shape)
print(df.dtypes)


## 2. Document counts by feed and duplicates

The RSS feed only returns a live snapshot (recent items), so running the
fetcher multiple times on different days will produce overlapping documents
across files. `document_id` should let us deduplicate.


In [ ]:
print("--- Documents per feed ---")
print(df["source_feed"].value_counts())

n_duplicates = df.duplicated(subset="document_id").sum()
print(f"\nDuplicate document_ids across fetch files: {n_duplicates}")

df_dedup = df.drop_duplicates(subset="document_id").reset_index(drop=True)
print(f"Documents after dedup: {len(df_dedup)} (was {len(df)})")


## 3. Publication date range

In [ ]:
df_dedup["pub_date_parsed"] = pd.to_datetime(
    df_dedup["pub_date"], format="mixed", errors="coerce"
)
print("Date range:", df_dedup["pub_date_parsed"].min(), "to", df_dedup["pub_date_parsed"].max())
print("\nUnparseable dates:", df_dedup["pub_date_parsed"].isna().sum())

df_dedup["pub_date_parsed"].value_counts().sort_index().plot(
    kind="line", marker="o", title="Documents by publication date"
)
plt.ylabel("count")
plt.tight_layout()
plt.show()


## 4. Document length distribution

This directly informs chunking (M4): the spec targets 300-500 tokens per
chunk. If most documents are already shorter than that, section-based
chunking may not even be necessary for the MVP — a document could be a
single chunk. If documents vary widely in length, that's a stronger signal
for real semantic/section-based chunking.


In [ ]:
df_dedup["word_count"] = df_dedup["clean_text"].str.split().str.len()

print(df_dedup["word_count"].describe())

df_dedup["word_count"].plot(
    kind="hist", bins=20, title="Document length (word count)", edgecolor="black"
)
plt.xlabel("words")
plt.tight_layout()
plt.show()


In [ ]:
# Longest and shortest documents -- worth eyeballing both extremes
print("Shortest documents:")
display(df_dedup.nsmallest(3, "word_count")[["title", "word_count"]])

print("\nLongest documents:")
display(df_dedup.nlargest(3, "word_count")[["title", "word_count"]])


## 5. Reference number extraction reliability

`rbi_fetcher.py` already extracts a reference number (e.g. `RBI/2026-27/248`)
per document via regex. Here we check how often that succeeds across the
whole corpus, and whether the failures follow a pattern (e.g. press releases
don't carry this format at all, which would be expected and fine).


In [ ]:
has_ref = df_dedup["reference_number"].notna()
print(f"Reference number found: {has_ref.sum()} / {len(df_dedup)} ({has_ref.mean():.0%})")

print("\nBy feed:")
print(df_dedup.groupby("source_feed")["reference_number"].apply(lambda s: s.notna().mean()))


## 6. Cross-reference density

The project's core premise is that multi-hop retrieval is *necessary*, not
decorative. This checks how many distinct RBI reference numbers appear
inside each document's own text (beyond its own reference number) --
evidence of the "this notification amends/refers to that other one" pattern
we saw in the sample data during M2.


In [ ]:
REFERENCE_PATTERN = re.compile(r"RBI/20\d{2}-\d{2}/\d+")

def other_reference_mentions(row):
    all_refs = set(REFERENCE_PATTERN.findall(row["clean_text"]))
    all_refs.discard(row["reference_number"])
    return len(all_refs)

df_dedup["n_other_references"] = df_dedup.apply(other_reference_mentions, axis=1)

print(df_dedup["n_other_references"].describe())
print(f"\nDocuments referencing at least one other RBI document: "
      f"{(df_dedup['n_other_references'] > 0).sum()} / {len(df_dedup)}")

df_dedup["n_other_references"].value_counts().sort_index().plot(
    kind="bar", title="Number of other RBI references mentioned per document"
)
plt.xlabel("# other reference numbers mentioned")
plt.ylabel("# documents")
plt.tight_layout()
plt.show()


## 7. Regulated entity category coverage

RBI directions are typically issued per regulated-entity type (Commercial
Banks, NBFCs, Co-operative Banks, Payments Banks, etc. -- the 11 categories
from the 2025 Master Direction consolidation). This is a simple keyword-based
detector, not a proper classifier -- good enough to see which categories our
current sample actually touches, which will shape the synthetic policy
corpus domains later (M7).


In [ ]:
ENTITY_CATEGORIES = {
    "Commercial Banks": r"Commercial Banks?",
    "Small Finance Banks": r"Small Finance Banks?",
    "Payments Banks": r"Payments? Banks?",
    "Regional Rural Banks": r"Regional Rural Banks?",
    "Local Area Banks": r"Local Area Banks?",
    "Urban Co-operative Banks": r"Urban Co-?operative Banks?",
    "Rural Co-operative Banks": r"Rural Co-?operative Banks?",
    "NBFC": r"Non-?Banking Financial Compan(?:y|ies)|NBFCs?",
    "All India Financial Institutions": r"All India Financial Institutions?",
    "Credit Information Companies": r"Credit Information Compan(?:y|ies)",
    "Asset Reconstruction Companies": r"Asset Reconstruction Compan(?:y|ies)",
}

def detect_categories(text):
    return [name for name, pattern in ENTITY_CATEGORIES.items()
            if re.search(pattern, text, re.IGNORECASE)]

df_dedup["entity_categories"] = (
    df_dedup["title"] + " " + df_dedup["clean_text"]
).apply(detect_categories)

category_counts = Counter(cat for cats in df_dedup["entity_categories"] for cat in cats)
category_series = pd.Series(category_counts).sort_values(ascending=False)
print(category_series)

if len(category_series) > 0:
    category_series.plot(kind="barh", title="Regulated entity category mentions")
    plt.xlabel("# documents")
    plt.tight_layout()
    plt.show()

n_uncategorized = (df_dedup["entity_categories"].str.len() == 0).sum()
print(f"\nDocuments matching no known category: {n_uncategorized} / {len(df_dedup)}")


## 8. Read a few full documents end-to-end

Numbers only go so far -- read a couple of complete documents to sanity-check
that `clean_text` is actually readable, coherent regulatory prose (not
mangled HTML remnants, truncated tables, etc.).


In [ ]:
for _, row in df_dedup.sample(min(3, len(df_dedup)), random_state=42).iterrows():
    print("=" * 100)
    print(f"TITLE: {row['title']}")
    print(f"REFERENCE: {row['reference_number']}  |  DATE: {row['pub_date']}  |  FEED: {row['source_feed']}")
    print("-" * 100)
    print(row["clean_text"])
    print()


## 9. Summary of findings

Fill this in after running the notebook against your actual fetched data.
Suggested prompts to answer here:

- **Document count & feed mix:** How many usable documents do we actually have?
  Is it enough to build a first retrieval pipeline against, or do we need to
  run the fetcher repeatedly over several days to accumulate more?
- **Length distribution:** Are documents short enough that per-document
  (not per-chunk) retrieval might work for the MVP, or is section-based
  chunking clearly needed?
- **Reference number reliability:** Does the regex reliably extract a
  reference number for notifications? Does it (correctly) fail for press
  releases?
- **Cross-reference density:** What fraction of documents reference another
  RBI document? Does this support treating multi-hop retrieval as a real
  requirement rather than a nice-to-have?
- **Entity category coverage:** Which regulated-entity categories are
  actually present in the sample? This should directly inform which domains
  the synthetic policy corpus (M7) covers first.
- **Any data quality issues spotted while reading full documents?**
